In [17]:
import numpy as np
import os
#import PIL
#import PIL.Image
import tensorflow as tf


In [18]:
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print(tf.config.list_physical_devices('GPU'))
print(tf.version.VERSION)

# Should print: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

Num GPUs Available:  0
[]
2.13.1


In [19]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

In [20]:
# Define parameters and data directory
batch_size = 32
img_height = 48
img_width = 48
num_classes = 7

train_dir = 'C:/Users/mpura/projects/emotion_detection_and_age_prediction/train'
test_dir = 'C:/Users/mpura/projects/emotion_detection_and_age_prediction/test'

### Data Preprocessing and Augmentation


In [21]:
train_ds = tf.keras.utils.image_dataset_from_directory(
  train_dir,
  validation_split=0.2,
  subset="training",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size)


Found 28709 files belonging to 7 classes.
Using 22968 files for training.


In [22]:
val_ds = tf.keras.utils.image_dataset_from_directory(
  train_dir,
  validation_split=0.2,
  subset="validation",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size)


Found 28709 files belonging to 7 classes.
Using 5741 files for validation.


In [23]:
from tensorflow.keras import layers

normalization_layer = layers.Rescaling(1./255)

data_augmentation = tf.keras.Sequential([
  layers.RandomFlip("horizontal"),
  layers.RandomRotation(0.1),
  layers.RandomZoom(0.1),
])

In [24]:
train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y))
train_ds = train_ds.map(lambda x, y: (tf.image.rgb_to_grayscale(x), y))
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))

val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (tf.image.rgb_to_grayscale(x), y))

### Model Building

In [25]:
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(48, 48, 1)),
    tf.keras.layers.Conv2D(64, kernel_size = (3,3), activation = 'relu'),

    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Dropout(0.25),

    tf.keras.layers.Conv2D(128, kernel_size=(3, 3), activation='relu'),
    tf.keras.layers.Conv2D(256, kernel_size=(3, 3), activation='relu'),

    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Dropout(0.25),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(1024, activation='relu'),
    tf.keras.layers.Dense(512, activation='relu'),

    tf.keras.layers.Dropout(0.25),

    tf.keras.layers.Dense(7, activation='softmax')
])

### Model Compliation and Training

In [26]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [27]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,                  # stop if val_loss doesn't improve for 5 epochs
        restore_best_weights=True    # revert to best checkpoint when stopped
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,                  # multiply LR by 0.5 when triggered
        patience=3,                  # trigger after 3 epochs of no improvement
        min_lr=1e-6,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath='best_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

In [29]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callbacks
)

Epoch 1/50
718/718 [==============================] - ETA: 0s - loss: 1.6741 - accuracy: 0.3329
Epoch 1: val_accuracy improved from 0.30378 to 0.38930, saving model to best_model.keras


ValueError: The following argument(s) are not supported with the native Keras format: ['options']